# Multi-GPU

Many GPU nodes are equipped with several GPUs, and many applications need more than one node.
There are two common ways of using multiple GPUs:
* **Multiple processes:** each process uses one GPU
* **Single process:** one process uses several GPUs

## Multiple Processes: MPI

MPI is the natural choice for distributed memory parallelism in HPC.
The most common setup is *one GPU per MPI rank*.
It scales beyond a single node and works with all GPU programming approaches.

Each rank has to select its GPU once at startup.
Often, the job scheduler or MPI launcher already restricts the GPUs visible to each rank, e.g. via `CUDA_VISIBLE_DEVICES`.
Otherwise, the node-local rank can be used together with the device selection mechanisms shown below.

The main challenge is the data exchange between ranks.
*CUDA-aware* MPI implementations accept device pointers directly:

```c++
MPI_Send(d_data, nx, MPI_DOUBLE, dest, tag, MPI_COMM_WORLD);
```

This requires obtaining raw device pointers (or managed memory pointers) from the respective approach.
The mechanics are the same as for calling CUDA kernels - see the [interoperability](./interop.ipynb) notebook.

For more information, check out the NHR@FAU courses
* [Introduction to Parallel Programming with MPI](https://go-nhr.de/course-mpi),
* [Hybrid Programming in HPC - MPI+X](https://go-nhr.de/course-mpi-x), and
* [Scaling CUDA-Accelerated Applications](https://go-nhr.de/course-scaling-cuda).

## Single Process: Multiple GPUs

Using several GPUs from one process follows the same pattern in all approaches that support it:
* Get the number of GPUs
* Loop over all GPUs
* Within the loop, select the current GPU and perform operations specific to it

Data allocations are specific to each GPU as well.
In practice, the data is usually split into one part per GPU.

We demonstrate this with a *hello world* example: each GPU prints its number and the thread index.
How many GPUs you see depends on your reservation - with a single GPU, all examples only print from GPU 0.

In [ ]:
!nvidia-smi -L

### OpenMP

`omp_get_num_devices` returns the number of GPUs, and the `device` clause selects the GPU for a single target construct.

```c++
int numDevices = omp_get_num_devices();

for (int d = 0; d < numDevices; ++d) {
#pragma omp target teams distribute parallel for device(d)
    for (size_t i0 = 0; i0 < nx; ++i0) {
        printf("Hello from GPU %d, thread %ld\n", d, i0);
    }
}
```

The complete example code is available in [hello-multi-gpu-omp-target.cpp](../src/multi-gpu/hello-multi-gpu-omp-target.cpp).

In [ ]:
!nvc++ -O3 -std=c++17 -mp=gpu -target=gpu -o ../build/multi-gpu/hello-multi-gpu-omp-target ../src/multi-gpu/hello-multi-gpu-omp-target.cpp

In [ ]:
!../build/multi-gpu/hello-multi-gpu-omp-target

Since target constructs synchronize implicitly at their end, the GPUs are used one after the other.
Adding `nowait` allows target constructs to execute asynchronously, i.e. all GPUs can work concurrently.
A `taskwait` then waits for all of them.

```c++
for (int d = 0; d < numDevices; ++d) {
#pragma omp target teams distribute parallel for device(d) nowait
    for (size_t i0 = 0; i0 < nx; ++i0) {
        // ...
    }
}
#pragma omp taskwait
```

### OpenACC

OpenACC compute constructs have no clause for selecting a GPU.
Instead, `acc_set_device_num` (or the equivalent `set device_num` directive) selects the GPU for all following constructs.

```c++
int numDevices = acc_get_num_devices(acc_device_nvidia);

for (int d = 0; d < numDevices; ++d) {
    acc_set_device_num(d, acc_device_nvidia);

#pragma acc parallel loop
    for (size_t i0 = 0; i0 < nx; ++i0) {
        printf("Hello from GPU %d, thread %ld\n", d, i0);
    }
}
```

The complete example code is available in [hello-multi-gpu-openacc.cpp](../src/multi-gpu/hello-multi-gpu-openacc.cpp).

In [ ]:
!nvc++ -O3 -std=c++17 -acc=gpu -target=gpu -o ../build/multi-gpu/hello-multi-gpu-openacc ../src/multi-gpu/hello-multi-gpu-openacc.cpp

In [ ]:
!../build/multi-gpu/hello-multi-gpu-openacc

As with OpenMP, compute constructs synchronize implicitly, and the GPUs are used one after the other.
With `async`, all GPUs can work concurrently.
Waiting then requires selecting each GPU again.

```c++
for (int d = 0; d < numDevices; ++d) {
    acc_set_device_num(d, acc_device_nvidia);

#pragma acc parallel loop async
    for (size_t i0 = 0; i0 < nx; ++i0) {
        // ...
    }
}

for (int d = 0; d < numDevices; ++d) {
    acc_set_device_num(d, acc_device_nvidia);
#pragma acc wait
}
```

### CUDA

`cudaSetDevice` selects the GPU for all following CUDA calls, including memory allocations and kernel launches.
Since kernel launches are asynchronous, all GPUs work concurrently.
Synchronization is per GPU.

```c++
int numDevices;
cudaGetDeviceCount(&numDevices);

for (int d = 0; d < numDevices; ++d) {
    cudaSetDevice(d);
    hello<<<1, nx>>>(d, nx);
}

for (int d = 0; d < numDevices; ++d) {
    cudaSetDevice(d);
    cudaDeviceSynchronize();
}
```

The complete example code is available in [hello-multi-gpu-cuda.cu](../src/multi-gpu/hello-multi-gpu-cuda.cu).

In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_86 -o ../build/multi-gpu/hello-multi-gpu-cuda ../src/multi-gpu/hello-multi-gpu-cuda.cu

In [ ]:
!../build/multi-gpu/hello-multi-gpu-cuda

### Thrust

Thrust algorithms run on the current CUDA device, i.e. `cudaSetDevice` applies as well.

```c++
for (int d = 0; d < numDevices; ++d) {
    cudaSetDevice(d);

    thrust::for_each(thrust::device, thrust::make_counting_iterator<size_t>(0), thrust::make_counting_iterator<size_t>(nx),
                     [=] __host__ __device__ (size_t i0) {
                         printf("Hello from GPU %d, thread %ld\n", d, i0);
                     });
}
```

The complete example code is available in [hello-multi-gpu-thrust.cu](../src/multi-gpu/hello-multi-gpu-thrust.cu).

In [ ]:
!nvcc -O3 -std=c++17 --extended-lambda -arch=sm_86 -o ../build/multi-gpu/hello-multi-gpu-thrust ../src/multi-gpu/hello-multi-gpu-thrust.cu

In [ ]:
!../build/multi-gpu/hello-multi-gpu-thrust

Thrust algorithms are synchronous, so the GPUs are used one after the other.
For concurrent execution, fall back to CUDA kernels or to CUB, whose algorithms are asynchronous and accept a CUDA stream.
Both are covered in the NHR@FAU course [Fundamentals of Accelerated Computing with Modern CUDA C++](https://go-nhr.de/course-dli-modern-cuda).

### Modern C++

Standard C++ has no notion of devices, so selecting a GPU is not supported.
Use MPI with one GPU per process as described above.

### Kokkos

Kokkos is geared towards using a single GPU per process, which is selected at initialization, e.g. with `Kokkos::InitializationSettings().set_device_id(...)`.
Using several GPUs from one process requires backend-specific code to create additional execution space instances - see the [Kokkos documentation](https://kokkos.org/kokkos-core-wiki/API/core/MultiGPUSupport.html).
Use MPI with one GPU per process as described above.

### SYCL

In SYCL, each queue is bound to a device.
Using several GPUs thus means creating one queue per GPU.

```c++
auto gpus = sycl::device::get_devices(sycl::info::device_type::gpu);

std::vector<sycl::queue> queues;
for (auto &gpu : gpus)
    queues.emplace_back(gpu);
```

Work submitted to different queues runs concurrently.

```c++
for (int d = 0; d < (int)queues.size(); ++d) {
    queues[d].submit([&](sycl::handler &h) {
        sycl::stream str(8192, 1024, h);
        h.parallel_for(nx, [=](auto i0) {
            str << "Hello from GPU " << d << ", thread " << (size_t)i0 << sycl::endl;
        });
    });
}

for (auto &q : queues)
    q.wait();
```

As in the [SYCL](./sycl.ipynb) notebook, printing requires compiling for the CUDA target.

The complete example code is available in [hello-multi-gpu-sycl.cpp](../src/multi-gpu/hello-multi-gpu-sycl.cpp).

In [ ]:
!acpp -O3 -march=native -std=c++17 --acpp-targets="cuda:sm_86" -o ../build/multi-gpu/hello-multi-gpu-sycl ../src/multi-gpu/hello-multi-gpu-sycl.cpp

In [ ]:
!../build/multi-gpu/hello-multi-gpu-sycl

## Next Step

Return to the [next steps](./next-steps.ipynb) notebook.